In [1]:
from google.colab import auth
auth.authenticate_user()

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [13]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
DATASET_ID = "produccion"
TABLE_ID= "Tabla_TiposTarjeta"

In [17]:
bigquery_client = bigquery.Client(project=PROJECT_ID)

schema_TipoT = [
        bigquery.SchemaField("DESCRIPCION_TARJETA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("BIN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MULTIBIN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_DE_TARJETA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_DESGRAVAMEN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA", bigquery.enums.SqlTypeNames.NUMERIC)
]

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    job_config = bigquery.LoadJobConfig()
    try:
      tabla = bigquery_client.get_table(table_ref)
    except:
        tabla_tramas = bigquery.Table(table_ref, schema=schema)
        tabla_tramas = bigquery_client.create_table(tabla_tramas)
        print(f'ℹ️ ----- Se ha creado la tabla: {table_id} en el dataset: {dataset_id} -----')
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    return


df_tipos_tarjeta= pd.read_excel('/content/drive/MyDrive/Bines y Multibines.xlsx', sheet_name='TASAS',dtype={'BIN': str})
df_tipos_tarjeta.columns = (df_tipos_tarjeta.columns
                            .str.strip()  # quitar espacios al inicio/fin
                            .str.upper()  # opcional: todo en mayúsculas
                            .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/_ por _
)

df_tipos_tarjeta['DESCRIPCION_TARJETA'] = df_tipos_tarjeta['DESCRIPCION_TARJETA'].str.strip().str.upper()
df_tipos_tarjeta['TIPO_DE_TARJETA'] = df_tipos_tarjeta['TIPO_DE_TARJETA'].str.strip().str.upper()
Guardar_en_BigQuery(df_tipos_tarjeta, DATASET_ID, TABLE_ID, schema_TipoT)
print(f"✅ ARCHIVO CARGADO ...")

✅ ARCHIVO CARGADO ...


In [4]:
df_tipos_tarjeta= pd.read_excel('/content/drive/MyDrive/Bines y Multibines.xlsx', sheet_name='TASAS',dtype={'BIN': str})

In [5]:
df_tipos_tarjeta.columns = (df_tipos_tarjeta.columns
                            .str.strip()  # quitar espacios al inicio/fin
                            .str.upper()  # opcional: todo en mayúsculas
                            .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/_ por _
)

In [6]:
df_tipos_tarjeta['DESCRIPCION_TARJETA'] = df_tipos_tarjeta['DESCRIPCION_TARJETA'].str.strip().str.upper()
df_tipos_tarjeta['TIPO_DE_TARJETA'] = df_tipos_tarjeta['TIPO_DE_TARJETA'].str.strip().str.upper()

In [7]:
df_tipos_tarjeta.head(3)

,DESCRIPCION_TARJETA,BIN,MULTIBIN,MONEDA,TIPO_DE_TARJETA,TIPO_DESGRAVAMEN,NUEVA_PRIMA
0,VISA LIFEMILES SIGNATURE,414089,V3,SOLES,VISA,PERSONA NATURAL,6.0
1,VISA LIFEMILES SIGNATURE,414089,V1,SOLES,VISA,PERSONA NATURAL,6.0
2,VISA LIFEMILES SIGNATURE,414089,V5,SOLES,VISA,PERSONA NATURAL,6.0


In [8]:
df_tipos_tarjeta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   DESCRIPCION_TARJETA  86 non-null     object 
 1   BIN                  86 non-null     object 
 2   MULTIBIN             86 non-null     object 
 3   MONEDA               86 non-null     object 
 4   TIPO_DE_TARJETA      86 non-null     object 
 5   TIPO_DESGRAVAMEN     86 non-null     object 
 6   NUEVA_PRIMA          86 non-null     float64
dtypes: float64(1), object(6)
memory usage: 4.8+ KB
